[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JoseGG8/AutoEncoderPi/blob/feature/circuitos-variacionales/modelo/comparativa_circuitos_qae.ipynb)

# 🔬 Comparativa de Circuitos Variacionales — Autoencoder Cuántico (QAE)
### Dataset Paderborn | Detección de Fallas en Rodamientos | Paradigma One-Class

---

## ¿Qué hace este notebook?
Implementa y **compara 7 ansätze (circuitos) variacionales** para el Autoencoder Cuántico.
Para cada circuito calcula:
- **Profundidad y ancho** (complejidad)
- **Visualización del circuito** con `qml.draw_mpl`
- **Curva de convergencia** (fidelidad vs época)
- **Error de reconstrucción** (fidelidad del trash qubit + MSE proxy)

---

## Arquitectura del QAE

```
|ψ_in⟩ (7 qubits, 128 amplitudes)
        │
   ┌────▼────────────────┐
   │   Ansätz  U(θ)      │  ← compuertas parametrizadas (se entrenan)
   └────┬──────────┬─────┘
        │          │
   K latentes   M trash qubits
   (info útil)  └─→ P(|0⟩) = Fidelidad
                    1 - F  = info perdida
```

---

## 📖 Glosario de Compuertas Cuánticas

| Compuerta | Qué hace | Params |
|-----------|----------|--------|
| **RY(θ)** | Rota en eje Y → cambia la **probabilidad** de medir \|0⟩ vs \|1⟩ | 1 |
| **RX(θ)** | Rota en eje X → similar a RY en dirección perpendicular | 1 |
| **RZ(φ)** | Rota en eje Z → cambia la **fase** (interferencia), no las probabilidades | 1 |
| **Rot(φ,θ,ω)** | = RZ·RY·RZ → rotación **arbitraria completa** en la esfera de Bloch | 3 |
| **H** | Hadamard → crea superposición 50-50: \|0⟩→(\|0⟩+\|1⟩)/√2 | 0 |
| **CNOT** | Si control=\|1⟩ aplica NOT al objetivo → **crea entrelazamiento** | 0 |
| **CZ** | Si ambos=\|1⟩ aplica fase -1 → más nativo en hardware superconductor | 0 |
| **IsingZZ(J)** | e^{-iJ·ZZ/2} → acoplamiento magnético cuántico entre vecinos | 1 |
| **IsingXX(J)** | e^{-iJ·XX/2} → acoplamiento en eje X, complementa al ZZ | 1 |


In [ ]:
# Instalar PennyLane si es necesario (descomenta en Colab)
# !pip install pennylane -q


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

import pennylane as qml
from pennylane import numpy as pnp

plt.style.use('seaborn-v0_8-whitegrid'
              if 'seaborn-v0_8-whitegrid' in plt.style.available else 'ggplot')
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11})
COLORES = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c','#e67e22']

print(f'PennyLane v{qml.__version__}')


---
## 📦 1. Carga y Preprocesamiento (reutilizado del pipeline principal)

Pipeline: **RobustScaler → Zero-Padding 128 dims → Normalización L₂**
División: **70% train / 15% val / 15% test** (fit del scaler solo en train)


In [ ]:
# Rutas posibles (local y Colab)
RUTAS = [
    '../../paderborn_features_tratadas.csv',
    '../paderborn_features_tratadas.csv',
    'paderborn_features_tratadas.csv',
    '/content/paderborn_features_tratadas.csv',
    os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'paderborn_features_tratadas.csv')),
]

ruta = next((r for r in RUTAS if os.path.exists(r)), None)
if ruta is None:
    raise FileNotFoundError('No se encontró paderborn_features_tratadas.csv')

df = pd.read_csv(ruta)
col_label   = 'Label' if 'Label' in df.columns else df.columns[-1]
feature_cols = [c for c in df.columns if c != col_label]
print(f'Dataset: {df.shape[0]} filas x {len(feature_cols)} features | {ruta}')

# ── Partición 70/15/15 ────────────────────────────────────────────────────
X_all = df[feature_cols].values
X_tr, X_tmp = train_test_split(X_all, test_size=0.30, random_state=42)
X_val, X_te = train_test_split(X_tmp, test_size=0.50, random_state=42)

# ── RobustScaler: fit SOLO en train (evita data leakage) ─────────────────
scaler = RobustScaler()   # Usa mediana e IQR → resistente a outliers
scaler.fit(X_tr)          # Aprende estadísticas solo de los datos de entrenamiento
X_tr  = scaler.transform(X_tr)
X_val = scaler.transform(X_val)
X_te  = scaler.transform(X_te)

# ── Zero-Padding a 128 dims (2^7 para Amplitude Encoding en 7 qubits) ────
def zero_pad(X, dim=128):
    """Extiende con ceros a la derecha hasta 'dim' dimensiones."""
    pad = dim - X.shape[1]
    return np.hstack([X, np.zeros((len(X), pad))]) if pad > 0 else X

X_tr  = zero_pad(X_tr);  X_val = zero_pad(X_val);  X_te = zero_pad(X_te)

# ── Normalización L2 por fila (condición cuántica: suma |a_i|^2 = 1) ──────
def l2(X):
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.where(n == 0, 1.0, n)  # Evita dividir por cero

X_train_q = l2(X_tr)
X_val_q   = l2(X_val)
X_test_q  = l2(X_te)

print(f'Train: {X_train_q.shape} | Val: {X_val_q.shape} | Test: {X_test_q.shape}')
print(f'Norma L2 media (debe ser ≈1): {np.linalg.norm(X_train_q, axis=1).mean():.6f}')


---
## ⛛️ 2. Los 7 Ansätze Variacionales

| # | Nombre | Idea central | Complejidad |
|---|--------|--------------|-------------|
| C1 | RY+RZ Circular *(baseline)* | 2 rotaciones + CNOT en anillo | Baja |
| C2 | BasicEntangler | Template PennyLane mínimo (RX + CNOT lineal) | Mínima |
| C3 | StronglyEntangling | Rot completo + CNOT no-local por capa | Media-alta |
| C4 | Hardware-Efficient | RY+RZ + CZ lineal (orientado a hardware real) | Media |
| C5 | Two-Design | Hadamard + RY + doble barrido CNOT | Media |
| C6 | Ising-Inspired | RX + IsingZZ + RZ (modelo de spins cuánticos) | Alta |
| C7 | Deep Alternating | Rot + doble CNOT + IsingXX (máxima expresividad) | Muy alta |


In [ ]:
# ==============================================================================
# CELDA 3: DEFINICIÓN DE LOS 7 ANSÄTZE
# ==============================================================================
TOTAL_QUBITS = 7  # 2^7 = 128 amplitudes

# ── C1: RY+RZ Circular (baseline) ─────────────────────────────────────────────
def ansatz_c1(weights, wires):
    """
    RY+RZ Circular — el más simple: 2 rotaciones por qubit + CNOT en anillo.
    Analogía: red neuronal de 1 capa con conexiones circulares.
    weights shape: (n_layers, n_wires, 2)
    """
    n = len(wires)
    for layer in weights:
        for i, w in enumerate(wires):
            qml.RY(layer[i, 0], wires=w)  # Ajusta P(|0>) vs P(|1>)
            qml.RZ(layer[i, 1], wires=w)  # Ajusta fase → interferencia
        for i in range(n - 1):
            qml.CNOT(wires=[wires[i], wires[i+1]])  # Entrelaza hacia adelante
        if n > 2:
            qml.CNOT(wires=[wires[-1], wires[0]])   # Cierra el anillo

def ws_c1(nl, nw): return (nl, nw, 2)  # 2 params/qubit/capa


# ── C2: BasicEntanglerLayers ───────────────────────────────────────────────────
def ansatz_c2(weights, wires):
    """
    BasicEntanglerLayers (template PennyLane).
    Solo RX (1 param/qubit) + CNOT lineal. Mínimo de parámetros.
    weights shape: (n_layers, n_wires)
    """
    qml.BasicEntanglerLayers(weights, wires=wires)  # RX + CNOT lineal automático

def ws_c2(nl, nw): return (nl, nw)  # 1 param/qubit/capa


# ── C3: StronglyEntanglingLayers ───────────────────────────────────────────────
def ansatz_c3(weights, wires):
    """
    StronglyEntanglingLayers (template PennyLane).
    Rot(φ,θ,ω) = RZ·RY·RZ (3 params/qubit) + CNOT con patrón variable.
    Crea correlaciones no-locales: qubits no adyacentes quedan entrelazados.
    weights shape: (n_layers, n_wires, 3)
    """
    qml.StronglyEntanglingLayers(weights, wires=wires)  # Rot + CNOT pattern

def ws_c3(nl, nw): return (nl, nw, 3)  # 3 params/qubit/capa


# ── C4: Hardware-Efficient Ansatz ──────────────────────────────────────────────
def ansatz_c4(weights, wires):
    """
    Hardware-Efficient Ansatz (HEA).
    Diseñado para hardware real (IBM Q, Google): usa CZ en vez de CNOT.
    CZ aplica fase -1 si ambos qubits=|1> → menos ruido en chips superconductores.
    Topología lineal (sin cerrar el anillo).
    weights shape: (n_layers, n_wires, 2)
    """
    n = len(wires)
    for layer in weights:
        for i, w in enumerate(wires):
            qml.RY(layer[i, 0], wires=w)  # Ajusta probabilidades
            qml.RZ(layer[i, 1], wires=w)  # Ajusta fase
        for i in range(n - 1):
            qml.CZ(wires=[wires[i], wires[i+1]])  # CZ: nativo en superconductores

def ws_c4(nl, nw): return (nl, nw, 2)


# ── C5: Simplified Two-Design ─────────────────────────────────────────────────
def ansatz_c5(weights, wires):
    """
    Simplified Two-Design.
    Inspirado en t-designs cuánticos: distribución aproximadamente Haar-random.
    Estructura: H inicial (superposición uniforme) → RY aprendido → CNOT alternados.
    El patrón par/impar de CNOTs asegura que todos los vecinos queden entrelazados.
    weights shape: (n_layers, n_wires)
    """
    n = len(wires)
    for w in wires:
        qml.Hadamard(wires=w)  # H|0> = (|0>+|1>)/√2 — incertidumbre máxima inicial
    for layer in weights:
        for i, w in enumerate(wires):
            qml.RY(layer[i], wires=w)      # Ajusta la superposición aprendida
        for i in range(0, n - 1, 2):      # Pares: (0,1),(2,3),(4,5)...
            qml.CNOT(wires=[wires[i], wires[i+1]])
        for i in range(1, n - 1, 2):      # Impares: (1,2),(3,4),(5,6)...
            qml.CNOT(wires=[wires[i], wires[i+1]])

def ws_c5(nl, nw): return (nl, nw)  # 1 param/qubit/capa


# ── C6: Ising-Inspired ────────────────────────────────────────────────────────
def ansatz_c6(weights, wires):
    """
    Ising-Inspired Ansatz.
    Basado en el modelo de Ising cuántico (física de spins magnéticos).
    Por capa: RX (campo transversal) → IsingZZ (acoplamiento J entre vecinos) → RZ.
    IsingZZ(J) = exp(-iJ·ZZ/2): dos spins interactuando como imanes adyacentes.
    weights shape: (n_layers, n_wires, 3)
    """
    n = len(wires)
    for layer in weights:
        for i, w in enumerate(wires):
            qml.RX(layer[i, 0], wires=w)          # Campo magnético transversal
        for i in range(n - 1):
            qml.IsingZZ(layer[i, 1], wires=[wires[i], wires[i+1]])
            # IsingZZ: acoplamiento cuántico entre spins vecinos
        for i, w in enumerate(wires):
            qml.RZ(layer[i, 2], wires=w)           # Ajuste de fase post-interacción

def ws_c6(nl, nw): return (nl, nw, 3)


# ── C7: Deep Alternating Ansatz ───────────────────────────────────────────────
def ansatz_c7(weights, wires):
    """
    Deep Alternating Ansatz (DAA) — el más complejo.
    Rot(φ,θ,ω) por qubit (3 params) + doble CNOT (forward+backward) + IsingXX.
    El doble barrido CNOT propaga info de TODOS los qubits a TODOS los demás.
    IsingXX(J) = exp(-iJ·XX/2): acoplamiento en eje X (complementa al IsingZZ).
    Mayor riesgo de Barren Plateaus con muchos qubits/capas.
    weights shape: (n_layers, n_wires, 4)
    """
    n = len(wires)
    for layer in weights:
        for i, w in enumerate(wires):
            qml.Rot(layer[i, 0], layer[i, 1], layer[i, 2], wires=w)
            # Rot = RZ·RY·RZ: orienta el qubit en cualquier punto de la esfera de Bloch
        for i in range(n - 1):                        # Forward: 0→1→2→...
            qml.CNOT(wires=[wires[i], wires[i+1]])
        for i in range(n - 1, 0, -1):                 # Backward: 6→5→4→...
            qml.CNOT(wires=[wires[i], wires[i-1]])    # Crea correlaciones de largo alcance
        for i in range(0, n - 1, 2):
            qml.IsingXX(layer[i, 3], wires=[wires[i], wires[i+1]])
            # IsingXX: acoplamiento en eje X entre pares alternos

def ws_c7(nl, nw): return (nl, nw, 4)  # 4 params/qubit/capa


# ── Registro de circuitos ─────────────────────────────────────────────────────
CIRCUITOS = [
    ('C1: RY+RZ Circular',     ansatz_c1, ws_c1),
    ('C2: BasicEntangler',     ansatz_c2, ws_c2),
    ('C3: StronglyEntangling', ansatz_c3, ws_c3),
    ('C4: Hardware-Efficient', ansatz_c4, ws_c4),
    ('C5: Two-Design',         ansatz_c5, ws_c5),
    ('C6: Ising-Inspired',     ansatz_c6, ws_c6),
    ('C7: Deep Alternating',   ansatz_c7, ws_c7),
]

print('Circuitos registrados:')
for nombre, _, wsfn in CIRCUITOS:
    s = wsfn(2, TOTAL_QUBITS)
    print(f'  {nombre:<28} shape={str(s):<18} params={int(__import__("numpy").prod(s))}')


---
## 📐 3. Profundidad, Ancho y Complejidad de cada Circuito

| Métrica | Descripción |
|---------|-------------|
| **Profundidad** | Capas secuenciales máximas. Mayor = más expresivo pero más ruido en hardware |
| **Ancho** | Número de qubits (siempre 7 aquí) |
| **Nº parámetros** | Parámetros entrenables (equivalente a los pesos de una red neuronal) |
| **Total puertas** | Operaciones cuánticas aplicadas |


In [ ]:
# ==============================================================================
# CELDA 4: MÉTRICAS DE COMPLEJIDAD — qml.specs()
# ==============================================================================

def calcular_metricas(nombre, ansatz_fn, ws_fn, n_layers=2, n_wires=7, trash=1):
    """Extrae profundidad, puertas y params usando qml.specs()."""
    dev_t = qml.device('default.qubit', wires=n_wires)
    tw    = list(range(n_wires - trash, n_wires))
    shape = ws_fn(n_layers, n_wires)
    w0    = np.zeros(shape)              # Pesos dummy solo para analizar estructura
    sv0   = np.ones(2**n_wires) / np.sqrt(2**n_wires)  # Estado uniforme de referencia

    @qml.qnode(dev_t)
    def circ(sv, w):
        qml.AmplitudeEmbedding(sv, wires=range(n_wires), normalize=False)
        ansatz_fn(w, wires=range(n_wires))
        return qml.probs(wires=tw)

    specs  = qml.specs(circ)(sv0, w0)
    res    = specs['resources']
    n_par  = int(np.prod(shape))

    return {
        'Circuito':      nombre,
        'Profundidad':   res.depth,
        'Ancho':         res.num_wires,
        'Nº Parámetros': n_par,
        'Total Puertas': res.num_gates,
        'Línea Base (%)': round(100.0 / (2**trash), 1),
    }

np.random.seed(0)
tabla_comp = [calcular_metricas(n, fn, ws) for n, fn, ws in CIRCUITOS]
df_comp    = pd.DataFrame(tabla_comp).set_index('Circuito')

print(df_comp.to_string())


In [ ]:
# ==============================================================================
# CELDA 5: GRÁFICA — PROFUNDIDAD, PARÁMETROS Y PUERTAS POR CIRCUITO
# ==============================================================================

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Complejidad de los 7 Circuitos QAE', fontsize=14, fontweight='bold')

etiq = [n.split(':')[0] for n in df_comp.index]  # C1, C2, ...
metr = ['Profundidad', 'Nº Parámetros', 'Total Puertas']
tits = ['Profundidad del Circuito', 'Parámetros Entrenables', 'Total de Puertas Cuánticas']
ylabs = ['Capas secuenciales', 'Nº parámetros', 'Nº puertas']

for ax, col, tit, yl in zip(axes, metr, tits, ylabs):
    bars = ax.bar(etiq, df_comp[col], color=COLORES, edgecolor='black', linewidth=0.6)
    ax.set_title(tit, fontweight='bold')
    ax.set_ylabel(yl)
    ax.set_xlabel('Circuito')
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2,
                str(int(b.get_height())), ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('complejidad_circuitos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: complejidad_circuitos.png')


---
## 🖼️ 4. Diagrama de cada Circuito

Cada línea horizontal = un qubit. Las cajas = compuertas. Las líneas verticales = entrelazamiento (CNOT, CZ, Ising...).

> Visualización con 2 capas y 7 qubits (1 trash qubit, K=6 latentes).


In [ ]:
# ==============================================================================
# CELDA 6: DIAGRAMAS DE CIRCUITO CON qml.draw_mpl()
# ==============================================================================

NL_VIZ, NW_VIZ, TR_VIZ = 2, 7, 1
np.random.seed(0)
sv_viz = np.ones(2**NW_VIZ) / np.sqrt(2**NW_VIZ)  # Estado uniforme de demo
tw_viz = list(range(NW_VIZ - TR_VIZ, NW_VIZ))

for idx, (nombre, ansatz_fn, ws_fn) in enumerate(CIRCUITOS):
    shape  = ws_fn(NL_VIZ, NW_VIZ)
    w_viz  = np.zeros(shape)
    dev_v  = qml.device('default.qubit', wires=NW_VIZ)

    @qml.qnode(dev_v)
    def circ_viz(sv, w, afn=ansatz_fn, tw=tw_viz, nw=NW_VIZ):
        qml.AmplitudeEmbedding(sv, wires=range(nw), normalize=False)
        afn(w, wires=range(nw))
        return qml.probs(wires=tw)

    fig, ax = qml.draw_mpl(circ_viz, style='pennylane')(sv_viz, w_viz)
    n_par   = int(np.prod(shape))
    fig.suptitle(f'{nombre}  |  Params: {n_par}  |  Capas: {NL_VIZ}',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    fname = f'circuito_{idx+1}_{nombre.split(":")[0]}.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Guardado: {fname}')


---
## 🏋️ 5. Entrenamiento Comparativo

Condiciones iguales para todos:
- **100 registros** de entrenamiento (Fase 1 — prototipo)
- **20 épocas**, batch 25, Adam lr=0.12
- **1 trash qubit** (K=6 latentes, línea base aleatoria = 50%)

> ⚠️ En CPU puede tardar varios minutos. En Google Colab con GPU es mucho más rápido.

### Función de costo
$$\mathcal{L}(\theta) = 1 - \mathbb{E}[P(|0\rangle^M_{\text{trash}})]$$

El objetivo es **maximizar la fidelidad** — que los trash qubits vuelvan a |0⟩, 
lo que significa que TODA la info relevante quedó en los K qubits latentes.


In [ ]:
# ==============================================================================
# CELDA 7: FUNCIÓN DE ENTRENAMIENTO REUTILIZABLE
# ==============================================================================

def entrenar_qae(nombre, ansatz_fn, ws_fn, X_tr, X_val,
                 n_layers=2, n_wires=7, trash=1,
                 epochs=20, batch=25, lr=0.12, n_samp=100, seed=42):
    """
    Entrena un QAE y devuelve historial de pérdidas/fidelidades.

    Retorna dict con: weights, qnode, fidelidad_fn, history, linea_base, n_params
    """
    np.random.seed(seed)
    dev   = qml.device('default.qubit', wires=n_wires)  # Simulador clásico de qubits
    tw    = list(range(n_wires - trash, n_wires))        # Índices de trash qubits
    shape = ws_fn(n_layers, n_wires)

    # Pesos iniciales aleatorios en [0, 2π] — requires_grad activa el cálculo de gradientes
    W = pnp.random.uniform(0.0, 2*np.pi, size=shape, requires_grad=True)

    @qml.qnode(dev)
    def qnode(sv, w):
        qml.AmplitudeEmbedding(sv, wires=range(n_wires), normalize=False)
        # AmplitudeEmbedding: codifica el vector clásico normalizado como estado cuántico
        ansatz_fn(w, wires=range(n_wires))   # Aplica el circuito variacional
        return qml.probs(wires=tw)            # P de cada estado posible en los trash qubits

    fid   = lambda sv, w: float(qnode(sv, w)[0])  # F = P(|0>) en trash

    def loss_b(w, B):
        return sum(1.0 - qnode(sv, w)[0] for sv in B) / len(B)  # 1 - F media

    X_demo = X_tr[:n_samp]
    opt    = qml.AdamOptimizer(stepsize=lr)  # Adam cuántico de PennyLane
    hist   = {'loss':[], 'fid_tr':[], 'fid_val':[]}
    lb     = 100.0 / (2**trash)

    print(f'\n[{nombre}]  params={int(np.prod(shape))}  trash={trash}  base={lb:.0f}%')

    for ep in range(epochs):
        perm = np.random.permutation(len(X_demo))
        ep_loss = []
        for i in range(0, len(X_demo), batch):
            B = X_demo[perm[i:i+batch]]
            W, cost = opt.step_and_cost(lambda w: loss_b(w, B), W)
            # step_and_cost: calcula gradiente y actualiza pesos en un solo paso
            ep_loss.append(float(cost))
        avg_l   = float(np.mean(ep_loss))
        fid_val = float(np.mean([fid(sv, W) for sv in X_val[:15]])) * 100
        hist['loss'].append(avg_l)
        hist['fid_tr'].append((1-avg_l)*100)
        hist['fid_val'].append(fid_val)
        if (ep+1) % 5 == 0 or ep == 0:
            print(f'  ep {ep+1:02d} | loss={avg_l:.4f} | '
                  f'train={hist["fid_tr"][-1]:.1f}% | val={fid_val:.1f}%')

    print(f'  Final → train={hist["fid_tr"][-1]:.1f}%  val={hist["fid_val"][-1]:.1f}%')
    return dict(nombre=nombre, weights=W, qnode=qnode, fid=fid,
                history=hist, lb=lb, n_params=int(np.prod(shape)), trash=trash)


In [ ]:
# ==============================================================================
# CELDA 8: ENTRENAR LOS 7 CIRCUITOS
# ==============================================================================
CFG = dict(n_layers=2, n_wires=7, trash=1, epochs=20,
           batch=25, lr=0.12, n_samp=100, seed=42)

resultados = {}
for nombre, afn, wsfn in CIRCUITOS:
    resultados[nombre] = entrenar_qae(nombre, afn, wsfn, X_train_q, X_val_q, **CFG)

print('\nEntrenamiento completo.')


---
## 📈 6. Curvas de Convergencia

La **línea punteada gris** es la fidelidad aleatoria pre-entrenamiento (50% para 1 trash qubit).
Un buen circuito sube sostenidamente por encima de esa línea.


In [ ]:
# ==============================================================================
# CELDA 9: GRÁFICA DE CONVERGENCIA (Train + Val)
# ==============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Convergencia de los 7 Circuitos QAE — Fidelidad por Época',
             fontsize=14, fontweight='bold')
lb_val = list(resultados.values())[0]['lb']

for (nombre, res), col in zip(resultados.items(), COLORES):
    h   = res['history']
    eps = range(1, len(h['fid_tr']) + 1)
    lbl = nombre.split(':')[0]
    ax1.plot(eps, h['fid_tr'],  'o-',  color=col, label=lbl, lw=1.8, ms=4)
    ax2.plot(eps, h['fid_val'], 's--', color=col, label=lbl, lw=1.8, ms=4)

for ax, tit in [(ax1,'Entrenamiento (Train)'), (ax2,'Validación (Val)')]:
    ax.axhline(lb_val, color='gray', ls=':', lw=1.5, label=f'Base aleatoria ({lb_val:.0f}%)')
    ax.set_xlabel('Época'); ax.set_ylabel('Fidelidad (%)'); ax.set_ylim(0, 105)
    ax.set_title(tit, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('convergencia_circuitos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: convergencia_circuitos.png')


---
## 🔍 7. Error de Reconstrucción: Fidelidad del Trash Qubit vs MSE

**¿Necesitamos un decoder explícito?** No es obligatorio. Aquí la explicación:

| Métrica | Fórmula | Significado |
|---------|---------|-------------|
| **Fidelidad** $F$ | $P(|0\rangle^M)$ | Qué tan bien se comprimió la info |
| **Error de reconstrucción** | $1 - F$ | Fracción de info perdida en los trash qubits |
| **MSE proxy** | $\overline{1-F}$ sobre el test set | Análogo al MSE input/output del decoder clásico |

La fidelidad del trash qubit **ya es** el error de reconstrucción cuántico.
`1 - F` mide exactamente cuánta información se 'fugó' a los qubits descartados.
Para conectar con el lenguaje del profesor, exponemos ambas métricas.


In [ ]:
# ==============================================================================
# CELDA 10: EVALUACIÓN — FIDELIDAD, ERROR DE RECONSTRUCCIÓN Y MSE PROXY
# ==============================================================================

N_EVAL = 50  # Muestras del test set para evaluación

# Anomalías sintéticas: inyectar perturbación espectral en componentes FFT
def hacer_anomalias(X, seed=7):
    np.random.seed(seed)
    Xan = X.copy()
    Xan[:, 10:25] += np.random.normal(0.4, 0.1, Xan[:, 10:25].shape)
    return l2(Xan)  # Re-normalizar tras perturbar

X_anom = hacer_anomalias(X_test_q[:N_EVAL])

rows = []
for nombre, res in resultados.items():
    fid_fn = res['fid']
    W      = res['weights']

    # Fidelidad en sanos (test set)
    fids_s = [fid_fn(sv, W) for sv in X_test_q[:N_EVAL]]
    fid_s  = np.mean(fids_s) * 100      # En porcentaje
    mse_s  = 1.0 - np.mean(fids_s)      # MSE proxy = 1 - F (entre 0 y 1)

    # Fidelidad en anomalías sintéticas
    fids_a = [fid_fn(sv, W) for sv in X_anom]
    fid_a  = np.mean(fids_a) * 100

    tau    = np.percentile([1-f for f in fids_s], 95)  # Umbral estadístico P95
    detec  = np.mean([(1-f) > tau for f in fids_a]) * 100  # Tasa detección

    rows.append({
        'Circuito':        nombre,
        'Fid. Sanos (%)':  round(fid_s, 2),
        'Fid. Anomalías (%)': round(fid_a, 2),
        'Separación (pp)': round(fid_s - fid_a, 2),
        'MSE Proxy (sanos)': round(mse_s, 4),
        'Detección (%)':   round(detec, 1),
    })
    print(f'  {nombre:<28} sanos={fid_s:.1f}%  anom={fid_a:.1f}%  '
          f'sep={fid_s-fid_a:.1f}pp  detec={detec:.0f}%')

df_eval = pd.DataFrame(rows).set_index('Circuito')
print("\n", df_eval.to_string())


In [ ]:
# ==============================================================================
# CELDA 11: HISTOGRAMAS — DISTRIBUCIÓN DE ERRORES SANOS vs ANOMALÍAS
# ==============================================================================

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
fig.suptitle('Distribución del Error de Reconstrucción (1 - Fidelidad)\n'
             'Verde=Sanos  |  Rojo=Anomalías  |  Negro=Umbral τ P95',
             fontsize=13, fontweight='bold')

for idx, (nombre, res) in enumerate(resultados.items()):
    ax = axes[idx]
    fid_fn = res['fid']; W = res['weights']

    err_s = [1 - fid_fn(sv, W) for sv in X_test_q[:N_EVAL]]
    err_a = [1 - fid_fn(sv, W) for sv in X_anom]
    tau   = np.percentile(err_s, 95)

    ax.hist(err_s, bins=12, alpha=0.65, color='#2ecc71', label='Sanos')
    ax.hist(err_a, bins=12, alpha=0.65, color='#e74c3c', label='Anomalías')
    ax.axvline(tau, color='black', ls='--', lw=1.5, label=f'τ={tau:.3f}')
    ax.set_title(nombre.split(':')[0], fontweight='bold')
    ax.set_xlabel('1 - Fidelidad'); ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=7)

axes[-1].axis('off')
plt.tight_layout()
plt.savefig('histogramas_anomalias.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: histogramas_anomalias.png')


In [ ]:
# ==============================================================================
# CELDA 12: TABLA RESUMEN + SCATTER — COMPLEJIDAD vs RENDIMIENTO
# ==============================================================================

df_final = df_comp[['Profundidad','Nº Parámetros','Total Puertas']].join(df_eval)
print("TABLA RESUMEN COMPLETA:")
print(df_final.to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Trade-off Complejidad vs Rendimiento', fontsize=14, fontweight='bold')

for (nombre, col) in zip(df_final.index, COLORES):
    r   = df_final.loc[nombre]
    etq = nombre.split(':')[0]
    ax1.scatter(r['Nº Parámetros'], r['Fid. Sanos (%)'], color=col, s=130, zorder=5)
    ax1.annotate(etq, (r['Nº Parámetros'], r['Fid. Sanos (%)']),
                 textcoords='offset points', xytext=(5,5), fontsize=9, fontweight='bold')
    ax2.scatter(r['Profundidad'], r['Separación (pp)'], color=col, s=130, zorder=5)
    ax2.annotate(etq, (r['Profundidad'], r['Separación (pp)']),
                 textcoords='offset points', xytext=(5,5), fontsize=9, fontweight='bold')

ax1.set_xlabel('Nº Parámetros', fontweight='bold')
ax1.set_ylabel('Fidelidad en Test — Sanos (%)', fontweight='bold')
ax1.set_title('¿Más parámetros = mejor fidelidad?')
ax2.set_xlabel('Profundidad del Circuito', fontweight='bold')
ax2.set_ylabel('Separación Sanos vs Anomalías (pp)', fontweight='bold')
ax2.set_title('¿Circuitos más profundos detectan mejor?')

plt.tight_layout()
plt.savefig('scatter_complejidad.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: scatter_complejidad.png')


---
## 🎓 8. Conclusiones y Recomendaciones

| Circuito | Cuándo usarlo |
|----------|---------------|
| C1: RY+RZ Circular | Baseline simple, fácil de explicar |
| C2: BasicEntangler | Mínimo de parámetros — máxima velocidad |
| C3: StronglyEntangling | **Mejor balance** expresividad/velocidad |
| C4: Hardware-Efficient | Para correr en hardware real (IBM Q) |
| C5: Two-Design | Exploración uniforme del espacio de estados |
| C6: Ising-Inspired | Datos con estructura de acoplamiento físico |
| C7: Deep Alternating | Máxima expresividad — riesgo de Barren Plateaus |

### Sobre la pregunta del decoder
- **`1 - Fidelidad`** del trash qubit ES el error de reconstrucción cuántico.
- No necesitas un decoder físico separado para medir cuánta info se pierde.
- El **MSE Proxy** calculado arriba conecta esta métrica con el lenguaje clásico.
- Recomendación final: **C3 StronglyEntangling** como circuito principal, **C1** como baseline comparativo.
